# Export one-group MGXS for the concentric reference reactor

This notebook reuses the concentric OpenMC reactor model from concentric_modeling.ipynb and keeps the transport-to-constants workflow in reusable Python code inside mgxs_export.py.

## Scope
- Start with a simple one-group constant set homogenized over the reactor fuel_element cell
- Export delayed neutron fractions and decay rates alongside the one-group transport constants
- Write machine-readable JSON and CSV outputs under build/notebook/export_mgxs/outputs

## Contract
- Use the concentric reactor model defined in concentric_modeling.ipynb
- Run with the openmc conda environment and configured OpenMC nuclear data
- Prefer reusable Python modules over copy-pasted notebook logic

## Note
The delayed neutron beta values exported here are one-group delayed neutron fractions from MGXS tallies. They are a useful starting point for point kinetics, but they are not the same as adjoint-weighted beta_eff from an IFP-style calculation.

In [12]:
import importlib
from dataclasses import asdict
from pprint import pprint
import sys

import openmc

import mgxs_export
from ploting import resolve_openmc_exec

importlib.reload(mgxs_export)

from mgxs_export import (
    MGXSExportConfig,
    build_directories,
    cross_sections_path,
    default_openmc_threads,
    reference_geometry_report,
    run_reference_mgxs_export,
)

In [9]:
EXPORT_CONFIG = MGXSExportConfig(
    particles=16000,
    batches=20,
    inactive=5,
)

EXPORT_PATHS = build_directories()
OPENMC_EXEC = resolve_openmc_exec()
OPENMC_THREADS = default_openmc_threads()
CROSS_SECTIONS = cross_sections_path()

NOTEBOOK_REPORT = {
    "python_executable": sys.executable,
    "openmc_version": openmc.__version__,
    "openmc_exec": OPENMC_EXEC,
    "openmc_threads": OPENMC_THREADS,
    "cross_sections": CROSS_SECTIONS,
    "reference_geometry": reference_geometry_report(),
    "export_config": asdict(EXPORT_CONFIG),
    "output_directory": str(EXPORT_PATHS["output_dir"]),
}

pprint(NOTEBOOK_REPORT, sort_dicts=False)

{'python_executable': '/home/pablo/miniconda3/envs/openmc/bin/python',
 'openmc_version': '0.15.3',
 'openmc_exec': '/home/pablo/miniconda3/envs/openmc/bin/openmc',
 'openmc_threads': 19,
 'cross_sections': PosixPath('/home/pablo/openmc/data/endfb-viii.1-hdf5/cross_sections.xml'),
 'reference_geometry': {'fuel_element': {'ring_count': 7,
                                         'ring_thickness_cm': 0.5,
                                         'coolant_gap_cm': 7.0,
                                         'inner_radius_cm': 4.5,
                                         'outer_radius_cm': 50.0,
                                         'control_rod_radius_cm': 4.0,
                                         'h_active_cm': 300.0,
                                         'lower_plenum_cm': 50.0,
                                         'upper_plenum_cm': 50.0,
                                         'fuel_density_g_per_cm3': 12.2,
                                         'fuel_enrichment_w

## Run the export

This cell launches an OpenMC eigenvalue calculation using the same concentric reference geometry as concentric_modeling.ipynb, attaches one-group MGXS tallies and delayed-neutron tallies to the reactor fuel_element cell, and writes JSON plus CSV outputs into the export build directory.

In [13]:
if not CROSS_SECTIONS:
    results = None
    print(
        "Set openmc.config['cross_sections'] or OPENMC_CROSS_SECTIONS and ensure the openmc conda environment is active before running this cell."
    )
else:
    results = run_reference_mgxs_export(
        config=EXPORT_CONFIG,
        base_dir=EXPORT_PATHS["root_dir"],
        threads=OPENMC_THREADS,
        openmc_exec=OPENMC_EXEC,
    )
    print(f"Statepoint: {results['statepoint_path']}")
    print(f"JSON export: {results['files']['json']}")
    print(f"Group constants CSV: {results['files']['group_constants_csv']}")
    print(f"Delayed neutron CSV: {results['files']['delayed_neutrons_csv']}")

/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=2.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=5.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=6.
  warn(msg, IDWarning)
/home/pablo/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=28.
  warn(msg, IDWarning)


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

## Summarize the one-group constants

Use this after the export cell to inspect the transport and delayed-neutron constants that were written to disk.

In [11]:
if results is None:
    print("Run the export cell first.")
else:
    run = results["run"]
    delayed = results["delayed_neutrons"]
    print(f"k_eff = {run['keff']['mean']:.6f} +/- {run['keff']['std_dev']:.6f}")
    print(f"reactivity = {run['reactivity_pcm']:.1f} pcm")

    generation_time = run.get("generation_time_s")
    if generation_time is not None:
        print(
            "generation time = "
            f"{generation_time['mean']:.6e} +/- {generation_time['std_dev']:.6e} s"
        )

    print(f"beta_total = {delayed['beta_total']:.6e}")
    if delayed["beta_weighted_decay_rate_per_s"] is not None:
        print(
            "beta-weighted decay rate = "
            f"{delayed['beta_weighted_decay_rate_per_s']:.6e} 1/s"
        )

    print("\nDelayed groups")
    for row in results["delayed_neutron_rows"]:
        print(
            f"group {row['delayed_group']}: "
            f"beta={row['beta']:.6e}, "
            f"lambda={row['decay_rate_per_s']:.6e} 1/s"
        )

    print("\nOne-group constants")
    for xs_type, payload in results["group_constants"].items():
        print(f"{xs_type:>22}: mean={payload['mean']} std_dev={payload['std_dev']}")

k_eff = 0.998421 +/- 0.002099
reactivity = -158.2 pcm
beta_total = 6.788467e-03
beta-weighted decay rate = 4.835494e-01 1/s

Delayed groups
group 1: beta=2.279177e-04, lambda=1.334430e-02 1/s
group 2: beta=1.195305e-03, lambda=3.267777e-02 1/s
group 3: beta=1.151922e-03, lambda=1.209145e-01 1/s
group 4: beta=2.624901e-03, lambda=3.041996e-01 1/s
group 5: beta=1.120549e-03, lambda=8.554025e-01 1/s
group 6: beta=4.678730e-04, lambda=2.872916e+00 1/s

One-group constants
                 total: mean=[0.4021635236404097] std_dev=[0.0018403179510750316]
             transport: mean=[0.34558528157088214] std_dev=[0.0018545874241597902]
            absorption: mean=[0.006244632789106411] std_dev=[2.424721631544655e-05]
 diffusion-coefficient: mean=[0.9645472510233749] std_dev=[0.005176254016445846]
            nu-fission: mean=[0.006861874569449435] std_dev=[2.941044385621096e-05]
         kappa-fission: mean=[546199.9456561998] std_dev=[2347.5637626622524]
                   chi: mean=[1.000